# DevAgent 项目 - Day 7：丰富工具系统 + Tool Integration

目标：
- 集成更多生产级工具（GitHub、Web Search、代码执行、文件操作等）
- 让 Multi-Agent 真正具备工程能力
- 优化工具调用体验
- 为后续产品级功能打基础

作者：Sun | Day 7

In [2]:
# ==================== 2. 配置环境 ====================
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

load_dotenv(override=True)

llm = ChatOllama(
    model="qwen2.5:14b",
    temperature=0.2,
    num_ctx=8192,
    num_gpu=999,
    base_url="http://host.docker.internal:11434"
)

print("✅ LLM 初始化完成")

✅ LLM 初始化完成


In [20]:
# ==================== 3. 定义丰富工具集 ====================

@tool
def web_search(query: str) -> str:
    """使用 Tavily 进行网页搜索"""
    print("===web_search")
    try:
        from tavily import TavilyClient
        client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
        response = client.search(query, max_results=5)
        results = [f"标题: {r['title']}\n内容: {r['content'][:300]}" for r in response.get('results', [])]
        return "\n\n".join(results) if results else "未找到相关信息"
    except Exception as e:
        return f"搜索失败: {str(e)}"
@tool
def github_search_repositories(query: str, max_results: int = 5) -> str:
    """模糊搜索 GitHub 仓库（推荐工具）"""
    try:
        from githubkit import GitHub
        gh = GitHub(token=os.getenv("GITHUB_TOKEN"))
        
        response = gh.rest.search.repos(
            q=query,
            per_page=max_results,
            sort="stars",
            order="desc"
        )
        repos = response.parsed_data.items
        
        if not repos:
            return "未找到匹配的仓库"
        
        result = []
        for repo in repos:
            result.append(f"""
仓库: {repo.full_name}
⭐ 星数: {repo.stargazers_count}
📝 描述: {repo.description or '无描述'}
🔗 URL: {repo.html_url}
语言: {repo.language}
            """.strip())
        
        return "\n\n".join(result)
    except Exception as e:
        return f"GitHub 搜索失败: {str(e)}（请确认 GITHUB_TOKEN 是否有效）"

@tool
def code_execution(code: str) -> str:
    """安全执行 Python 代码（简单版）"""
    print("===code_execution")
    try:
        allowed_globals = {"__builtins__": {}}
        result = eval(code, allowed_globals)
        return str(result)
    except Exception as e:
        return f"执行错误: {str(e)}"

@tool
def list_files(directory: str = ".") -> str:
    """列出指定目录下的文件"""
    print("===list_files")
    import os
    try:
        files = os.listdir(directory)
        return "\n".join(files[:20])  # 限制数量
    except Exception as e:
        return f"目录读取失败: {str(e)}"

@tool
def read_file(file_path: str) -> str:
    """读取文件内容"""
    print("===read_file")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()[:2000]  # 限制长度
    except Exception as e:
        return f"读取文件失败: {str(e)}"

tools = [web_search, github_search_repo, code_execution, list_files, read_file]

print(f"✅ 已定义 {len(tools)} 个实用工具")

✅ 已定义 5 个实用工具


In [21]:
# ==================== 4. 创建带工具的 Researcher Agent ====================
from langchain.agents import create_agent

researcher_system_prompt = """你是一位专业的研究助手。
你可以调用工具来获取最新信息、代码执行结果、文件内容等。
回答要准确、专业、结构化。"""

researcher_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=researcher_system_prompt
)

print("✅ Researcher Agent（带丰富工具）创建完成")

✅ Researcher Agent（带丰富工具）创建完成


In [22]:
# ==================== 5. 测试单个工具调用 ====================
from langchain_core.messages import HumanMessage

test_queries = [
    # "帮我搜索一下 LangGraph 最新发展",
    # "列出当前目录下的主要文件",
    # "执行代码：print(9876 * 543)",
    "langchain在github上的repo链接是什么，有多少个stars"
]

for q in test_queries:
    print(f"\n🔍 测试问题: {q}")
    result = researcher_agent.invoke({"messages": [HumanMessage(content=q)]})
    final_msg = result["messages"][-1]
    print(final_msg.content[:500] + "..." if len(final_msg.content) > 500 else final_msg.content)


🔍 测试问题: langchain在github上的repo链接是什么，有多少个stars
===github_search_repo
Response(404 Not Found, data_model=<class 'githubkit.versions.v2026_03_10.models.group_0006.BasicError'>)
===web_search
===github_search_repo
Response(404 Not Found, data_model=<class 'githubkit.versions.v2026_03_10.models.group_0006.BasicError'>)
看来直接通过工具调用未能成功检索到`langchain-ai/langchain`的信息。不过，根据之前的搜索结果，我们可以手动确认这个仓库的链接和star数量。

让我们访问[langchain-ai/langchain](https://github.com/langchain-ai/langchain)页面来查看其具体信息。
由于直接获取数据失败了，你可以通过点击上述链接进入GitHub页面自行查看该仓库的具体star数量。通常情况下，你可以在仓库主页顶部看到star的数量显示。


In [7]:
# ==================== 6. Day 7 总结 ====================
print("""
=== Day 7 完成总结 ===

✅ 已完成：
1. 集成多个生产级工具（Web Search、GitHub、Code Execution、File I/O）
2. Researcher Agent 支持丰富工具调用
3. 工具调用调试与测试

下一天计划（Day 8）：
- Human-in-the-Loop（人工确认关键操作）
- 增强 Reflection（LLM-based）
- 状态持久化（SQLiteSaver）
- 结构化输出（Pydantic）

项目已具备较强的工程能力！
""")


=== Day 7 完成总结 ===

✅ 已完成：
1. 集成多个生产级工具（Web Search、GitHub、Code Execution、File I/O）
2. Researcher Agent 支持丰富工具调用
3. 工具调用调试与测试

下一天计划（Day 8）：
- Human-in-the-Loop（人工确认关键操作）
- 增强 Reflection（LLM-based）
- 状态持久化（SQLiteSaver）
- 结构化输出（Pydantic）

项目已具备较强的工程能力！

